# Predicción de cancelación de clientes

**Caso base de la tarea — Curso de Despliegue y Producción de Modelos de ML**

---

## El problema

Una empresa de suscripciones quiere saber **qué clientes están por darse de baja**,
para poder contactarlos antes de que se vayan.

Tenemos 6.000 clientes con su historial, y sabemos cuáles cancelaron. Vamos a
entrenar un modelo que, dado un cliente, estime **la probabilidad de que cancele**.

## Qué hace este notebook

1. Carga y explora los datos
2. Arma un pipeline de preprocesamiento + modelo
3. Entrena
4. **Evalúa el modelo con detalle** — esta es la parte larga, y a propósito
5. Prueba el modelo con un cliente nuevo

## Qué NO hace este notebook

No guarda el modelo. **Eso es tu tarea**, y está al final, en la sección marcada
`TAREA`. Leé todo el notebook antes de llegar ahí: vas a necesitar entender qué
produce cada paso.


## 1. Configuración

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (11, 4)

print("pandas ", pd.__version__)
print("numpy  ", np.__version__)
print("sklearn", sklearn.__version__)
print("joblib ", joblib.__version__)

## 2. Los datos

Cada fila es un cliente. La columna `cancelo` es lo que queremos predecir:
vale 1 si el cliente se dio de baja y 0 si sigue activo.

In [ ]:
clientes = pd.read_csv("clientes.csv")

print("Filas:", len(clientes))
print("Columnas:", list(clientes.columns))
clientes.head()

### Qué significa cada columna

| Columna | Tipo | Qué es |
|---|---|---|
| `antiguedad_meses` | número | meses que lleva como cliente |
| `gasto_mensual` | número | cuánto paga por mes |
| `visitas_ultimo_mes` | número | veces que usó el servicio en el último mes |
| `dias_desde_ultima_visita` | número | días desde la última vez que lo usó |
| `tickets_soporte` | número | reclamos abiertos en el último mes |
| `plan` | texto | `basico`, `estandar` o `premium` |
| `metodo_pago` | texto | `tarjeta`, `transferencia` o `efectivo` |
| `descuento_activo` | 0 o 1 | si hoy tiene un descuento aplicado |
| **`cancelo`** | 0 o 1 | **lo que queremos predecir** |

In [ ]:
clientes.describe().round(2).T

### ¿Está balanceado?

In [ ]:
print(clientes.cancelo.value_counts())
print()
print(f"Tasa de cancelación: {clientes.cancelo.mean():.1%}")

Un 32% de cancelación. **No está balanceado, pero tampoco es un caso extremo.**

Esto importa para elegir la métrica: si el modelo dijera "nadie cancela" para
todos, acertaría el 68% de las veces. Por eso la *accuracy* sola no alcanza, y
más abajo vamos a mirar precisión, recall y AUC.

## 3. Exploración: ¿qué separa a los que cancelan?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))

clientes.groupby("plan").cancelo.mean().plot.bar(ax=ax[0], color="#4C78A8")
ax[0].set_title("Cancelación por plan"); ax[0].set_ylabel("tasa"); ax[0].set_xlabel("")

clientes.groupby("metodo_pago").cancelo.mean().plot.bar(ax=ax[1], color="#F58518")
ax[1].set_title("Cancelación por método de pago"); ax[1].set_xlabel("")

clientes.boxplot(column="dias_desde_ultima_visita", by="cancelo", ax=ax[2])
ax[2].set_title("Días sin usar el servicio"); ax[2].set_xlabel("canceló")
plt.suptitle("")
plt.tight_layout(); plt.show()

**Lectura:**

- Quien paga en **efectivo** cancela mucho más que quien paga con tarjeta. Tiene
  sentido: el pago manual da una oportunidad de decidir no pagar todos los meses.
- **Premium** retiene mejor que los otros planes.
- Los que cancelaron llevaban **más días sin usar el servicio**. Es la señal más
  fuerte del dataset, y también la más accionable.

In [ ]:
# Correlación de las variables numéricas con la cancelación
numericas = ["antiguedad_meses", "gasto_mensual", "visitas_ultimo_mes",
             "dias_desde_ultima_visita", "tickets_soporte", "descuento_activo"]

correlaciones = clientes[numericas + ["cancelo"]].corr()["cancelo"].drop("cancelo")
correlaciones.sort_values().plot.barh(color="#54A24B", figsize=(8, 3))
plt.title("Correlación con la cancelación"); plt.tight_layout(); plt.show()

correlaciones.sort_values(ascending=False).round(3)

## 4. Separar los datos

`X` son las variables de entrada y `y` es lo que queremos predecir.

Fijate en que **`cancelo` no está en `X`**. Si estuviera, el modelo tendría la
respuesta y aprendería nada.

In [ ]:
COLUMNAS_NUMERICAS = ["antiguedad_meses", "gasto_mensual", "visitas_ultimo_mes",
                      "dias_desde_ultima_visita", "tickets_soporte", "descuento_activo"]
COLUMNAS_CATEGORICAS = ["plan", "metodo_pago"]

COLUMNAS = COLUMNAS_NUMERICAS + COLUMNAS_CATEGORICAS

X = clientes[COLUMNAS]
y = clientes["cancelo"]

X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y,          # mantiene la misma proporción de cancelaciones en las dos partes
)

print(f"Entrenamiento: {len(X_entrena):,} clientes")
print(f"Prueba       : {len(X_prueba):,} clientes")
print(f"\nTasa de cancelación — entrenamiento: {y_entrena.mean():.1%}, prueba: {y_prueba.mean():.1%}")

> **Por qué `stratify=y`:** sin eso, el azar podría dejar 40% de cancelaciones en
> una parte y 25% en la otra, y la evaluación mediría el desbalance en vez del
> modelo.

## 5. El pipeline

Las columnas necesitan tratamientos distintos:

- las **numéricas** se escalan, para que `gasto_mensual` (que llega a 58) no pese
  más que `tickets_soporte` (que llega a 5) solo por la magnitud
- las **categóricas** (texto) hay que convertirlas a números, con *one-hot*:
  `plan` se convierte en tres columnas de 0 y 1

Un `Pipeline` junta el preprocesamiento y el modelo en **un solo objeto**.

> **Esto es lo más importante del notebook para tu tarea.** Como el pipeline
> incluye el preprocesamiento, cuando lo guardes y lo cargues en la API vas a
> poder pasarle el cliente **con los textos tal como vienen** (`plan="premium"`),
> sin tener que repetir la conversión a números. Si el preprocesamiento estuviera
> fuera del pipeline, tendrías que reproducirlo exactamente en la API — y ahí es
> donde se rompen los modelos en producción.

In [ ]:
preprocesamiento = ColumnTransformer([
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
])

modelo = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42)),
])

modelo

`handle_unknown="ignore"` significa: si en producción llega un plan que el
modelo nunca vio, no explota. Lo trata como si todas las categorías fueran cero.

No es lo ideal —conviene rechazarlo en la API, y eso es parte de tu tarea— pero
evita que el servicio se caiga.

## 6. Entrenar

In [ ]:
modelo.fit(X_entrena, y_entrena)

print("Modelo entrenado.")

## 7. Evaluación

Acá está el trabajo real. Un modelo no se juzga con un solo número.

### 7.1 Las cuatro métricas básicas

In [ ]:
# predict() da la clase (0 o 1); predict_proba() da la probabilidad
prediccion = modelo.predict(X_prueba)
probabilidad = modelo.predict_proba(X_prueba)[:, 1]     # columna 1 = probabilidad de cancelar

exactitud = accuracy_score(y_prueba, prediccion)
precision = precision_score(y_prueba, prediccion)
sensibilidad = recall_score(y_prueba, prediccion)
f1 = f1_score(y_prueba, prediccion)
auc = roc_auc_score(y_prueba, probabilidad)

print(f"Accuracy  : {exactitud:.4f}   de cada 100 clientes, acierta en {exactitud*100:.0f}")
print(f"Precision : {precision:.4f}   de los que marca como 'va a cancelar', acierta {precision*100:.0f}%")
print(f"Recall    : {sensibilidad:.4f}   de los que realmente cancelan, detecta {sensibilidad*100:.0f}%")
print(f"F1        : {f1:.4f}   el equilibrio entre las dos anteriores")
print(f"ROC-AUC   : {auc:.4f}   qué tan bien separa a los dos grupos")

### 7.2 Qué significa cada una, en este caso

| Métrica | Qué responde | Por qué importa acá |
|---|---|---|
| **Accuracy** | ¿Cuántos acertó en total? | Engaña con datos desbalanceados: decir "nadie cancela" da 68% |
| **Precision** | De los que marqué, ¿cuántos cancelaron? | Si es baja, el equipo comercial pierde tiempo llamando a gente que no se iba a ir |
| **Recall** | De los que cancelaron, ¿cuántos detecté? | Si es bajo, se te escapan los clientes que querías salvar |
| **ROC-AUC** | ¿Ordena bien por riesgo? | No depende del umbral. Es la métrica más honesta para comparar modelos |

**Precision y recall se pelean entre sí.** Si bajás el umbral, marcás más gente:
detectás más cancelaciones (recall sube) pero te equivocás más (precision baja).
Cuál conviene depende de cuánto cuesta llamar a un cliente contra cuánto vale
retenerlo.

### 7.3 La matriz de confusión

In [ ]:
matriz = confusion_matrix(y_prueba, prediccion)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matriz, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{matriz[i, j]:,}", ha="center", va="center", fontsize=15)
ax.set_xticks([0, 1], ["Predijo: sigue", "Predijo: cancela"])
ax.set_yticks([0, 1], ["Real: sigue", "Real: cancela"])
ax.set_title("Matriz de confusión")
plt.tight_layout(); plt.show()

verdaderos_negativos, falsos_positivos, falsos_negativos, verdaderos_positivos = matriz.ravel()
print(f"Acertó que seguían  : {verdaderos_negativos:,}")
print(f"Acertó que cancelan: {verdaderos_positivos:,}")
print(f"Falsa alarma       : {falsos_positivos:,}  (los marcó y no se iban)")
print(f"Se le escaparon    : {falsos_negativos:,}  (cancelaron y no los detectó)")

### 7.4 El reporte completo

In [ ]:
print(classification_report(y_prueba, prediccion,
                            target_names=["Sigue activo", "Cancela"], digits=3))

### 7.5 La curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_prueba, probabilidad)

plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, lw=2, label=f"Modelo (AUC = {auc:.3f})", color="#4C78A8")
plt.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="Azar (AUC = 0.500)")
plt.xlabel("Falsos positivos"); plt.ylabel("Verdaderos positivos")
plt.title("Curva ROC"); plt.legend(); plt.tight_layout(); plt.show()

Cuanto más se despega la curva de la diagonal, mejor separa el modelo.

Una referencia práctica para leer el AUC: 0.5 es azar, 0.7 es aceptable, 0.8 es
bueno y arriba de 0.9 es muy bueno **para este tipo de problema**. Un AUC de 0.99
en datos de negocio suele ser señal de que se coló una variable que no debería
estar ahí.

### 7.6 ¿Y si cambiamos el umbral?

Por defecto, `predict()` marca como "cancela" a todo el que supere 0.5 de
probabilidad. Ese número **es una decisión de negocio, no del modelo**.

In [ ]:
print(f"{'umbral':>8}{'marcados':>10}{'precision':>11}{'recall':>9}{'F1':>8}")
for umbral in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_umbral = (probabilidad >= umbral).astype(int)
    print(f"{umbral:>8.1f}{pred_umbral.sum():>10,}"
          f"{precision_score(y_prueba, pred_umbral, zero_division=0):>11.3f}"
          f"{recall_score(y_prueba, pred_umbral):>9.3f}"
          f"{f1_score(y_prueba, pred_umbral):>8.3f}")

Mirá cómo se mueven: **bajar el umbral detecta más cancelaciones pero genera
más falsas alarmas.**

Si el equipo comercial puede llamar a 500 clientes por mes, elegís el umbral que
te dé 500 marcados. Si llamar es caro, subís el umbral para equivocarte menos.

Guardá este número: **el umbral va a ser parte del bundle de tu tarea.**

### 7.7 ¿El resultado es estable?

Una sola partición puede haber tenido suerte. La validación cruzada entrena cinco
veces con particiones distintas.

In [ ]:
puntajes = cross_val_score(modelo, X, y, cv=5, scoring="roc_auc")

print("AUC en cada partición:", puntajes.round(4))
print(f"\nPromedio: {puntajes.mean():.4f}  (±{puntajes.std():.4f})")

Una desviación chica significa que el resultado no depende de cómo partimos los
datos. Si variara mucho, no podríamos confiar en el número del conjunto de prueba.

### 7.8 ¿Qué mira el modelo?

Como es una regresión logística, los coeficientes se pueden leer: positivo
empuja hacia cancelar, negativo hacia quedarse.

In [ ]:
nombres = modelo.named_steps["preprocesamiento"].get_feature_names_out()
coeficientes = modelo.named_steps["clasificador"].coef_[0]

importancia = pd.Series(coeficientes, index=nombres).sort_values()

importancia.plot.barh(figsize=(9, 5),
                      color=["#E45756" if c > 0 else "#4C78A8" for c in importancia])
plt.title("Coeficientes — rojo empuja a cancelar, azul a quedarse")
plt.axvline(0, color="black", lw=0.8)
plt.tight_layout(); plt.show()

importancia.sort_values(key=abs, ascending=False).round(3)

**Coincide con lo que vimos en la exploración**, y eso es buena señal: pagar en
efectivo y llevar días sin usar el servicio empujan a cancelar; la antigüedad y
las visitas retienen.

Cuando un modelo "aprende" algo que contradice el negocio, casi siempre hay un
problema en los datos.

## 8. Probarlo con un cliente nuevo

Así se usa el modelo, y así lo va a usar tu API.

In [ ]:
cliente_nuevo = pd.DataFrame([{
    "antiguedad_meses": 10,
    "gasto_mensual": 14.0,
    "visitas_ultimo_mes": 4,
    "dias_desde_ultima_visita": 26,
    "tickets_soporte": 1,
    "plan": "basico",
    "metodo_pago": "efectivo",
    "descuento_activo": 0,
}])

prob = modelo.predict_proba(cliente_nuevo)[0, 1]

print(f"Probabilidad de cancelar: {prob:.1%}")
print("Predicción:", "CANCELA" if prob >= 0.5 else "sigue activo")

Fijate en dos cosas, porque las vas a necesitar:

1. La entrada es un **DataFrame de una fila**, con las columnas por su nombre.
   El pipeline se encarga del resto: no hay que escalar ni convertir a números.
2. `predict_proba(...)[0, 1]` es **la probabilidad de la clase 1**. El `[0]` es la
   primera (y única) fila; el `[1]` es la columna de "cancela".

In [ ]:
# Un cliente con el perfil opuesto
cliente_fiel = pd.DataFrame([{
    "antiguedad_meses": 20,
    "gasto_mensual": 45.0,
    "visitas_ultimo_mes": 8,
    "dias_desde_ultima_visita": 10,
    "tickets_soporte": 0,
    "plan": "premium",
    "metodo_pago": "tarjeta",
    "descuento_activo": 0,
}])

print(f"Probabilidad de cancelar: {modelo.predict_proba(cliente_fiel)[0, 1]:.1%}")

---
---

# TAREA

Hasta acá llega el notebook que te entregamos. **Lo que sigue lo escribís vos.**

El modelo está entrenado y evaluado, pero vive en la memoria de este notebook:
cuando lo cierres, se pierde.

Tu tarea tiene tres partes, y están descritas en detalle en el documento
**`TAREA_bundle_y_api.md`**:

1. **Construir el bundle** — acá abajo, en este notebook
2. **Guardarlo y verificarlo** — acá abajo también
3. **Construir la API** — en archivos `.py` aparte

Leé el documento antes de empezar. Abajo quedan las celdas donde va la parte 1 y 2.


## TAREA · Parte 1 — Construir el bundle

Un diccionario con el modelo **y todo lo que la API va a necesitar saber**.

Repasá el documento de la tarea para ver qué claves tiene que tener y de dónde
sale cada valor. Casi todos los valores ya están calculados más arriba en este
notebook: buscalos, no los inventes.

In [ ]:
# TAREA — Parte 1: armá el diccionario `bundle`

bundle = {
    # completar
}

bundle

## TAREA · Parte 2 — Guardarlo y verificarlo

Guardar con `joblib.dump`, y después **volver a cargarlo y predecir**, para
comprobar que el archivo sirve de verdad.

In [ ]:
# TAREA — Parte 2a: guardar el bundle en "modelo_churn.joblib"


# TAREA — Parte 2b: cargarlo de nuevo y predecir con `cliente_nuevo`.
# La probabilidad tiene que dar EXACTAMENTE la misma que arriba.

## TAREA · Parte 3 — La API

Esa parte no va en el notebook. Se hace en archivos `.py`, y está explicada en
el documento de la tarea.

> Antes de pasar a la parte 3, asegurate de que la parte 2 funcione: si el
> archivo no se puede cargar y predecir, la API no tiene con qué trabajar.
